In [ ]:
import einops
import torch

from einops.layers.torch import EinMix
from utils.config import *
from utils.components import *
from utils.loss_fn import *
from utils.random_fields import RandomField

class EinMask_ENS(torch.nn.Module):
    def __init__(self, network: NetworkConfig, world: WorldConfig):
        super().__init__()
        # config attributes
        self.network = network
        self.world = world

        # default output dimension
        DO = default(network.dim_out, network.dim)

        # learnable parameters
        self.latent_tokens = torch.nn.Parameter(
            torch.nn.init.trunc_normal_(torch.zeros(network.num_latents, network.dim), std = network.dim ** -0.5)
            )
        self.position_codes = torch.nn.Parameter(init_sincos_positions(network.dim, shape = world.token_shape))
        
        # noise generator
        self.random_field = RandomField(network, world)

        # I/O
        self.to_tokens = torch.nn.Sequential(
            EinMix(f'b {world.field_pattern} -> b ({world.token_pattern}) c',
                weight_shape = f'v {world.patch_pattern} c', 
                c = network.dim, **world.token_sizes, **world.patch_sizes),
            torch.nn.RMSNorm(network.dim)
        )

        self.to_output = torch.nn.Sequential(
            EinMix(f'b ({world.token_pattern}) d -> b {world.field_pattern}',
                   weight_shape = f'v {world.patch_pattern} d',
                   d = DO, **world.patch_sizes, **world.token_sizes),
            #GaussianSmoothing3D(world.field_shape[0], kernel_size= 5, sigma= 1.),
        )

        self.to_decoder = torch.nn.Sequential(
            torch.nn.Linear(network.dim, DO, bias = False),
            torch.nn.RMSNorm(DO)
        )
        
        # Encoder / Decoder
        self.encoder = torch.nn.ModuleList([
                TransformerBlock(dim= network.dim, num_heads= network.num_encoder_heads, drop_path= network.drop_path) 
                for _ in range(default(network.num_read_blocks, 1))
                ])
        
        self.decoder = torch.nn.ModuleList([
                TransformerBlock(dim= DO, num_heads= network.num_decoder_heads, dim_kv= network.dim) 
                for _ in range(default(network.num_write_blocks, 1))
                ])
        
        # weight initialization
        self.apply(self.base_init)
        
    def base_init(self, m: torch.nn.Module):
        if isinstance(m, EinMix) or isinstance(m, torch.nn.Linear):
            torch.nn.init.trunc_normal_(m.weight, std = m.weight.size(-1) ** -0.5)
            if exists(m.bias):
                torch.nn.init.zeros_(m.bias)
   
    def forward(self, fields: torch.FloatTensor, visible: torch.BoolTensor, rng: torch.Generator = None) -> torch.FloatTensor:
        # expand position codes
        coo = einops.repeat(self.position_codes, '... -> b ...', b = fields.size(0))

        # sample random noise
        xi = self.random_field(tokens, rng)

        # tokenize and select visible
        obs = self.to_tokens(fields) + xi + coo
        obs = einops.rearrange(obs[visible], '(b m) ... -> b m ...', b = fields.size(0))
        
        # pad with latent tokens
        src = einops.repeat(self.latent_tokens, 'z d -> b z d', b = fields.size(0))
        src, ps = einops.pack([obs, src], 'b * d')

        # self-attention encoder
        for read in self.encoder:
            src = read(src)

        # optional bottleneck
        if self.network.kwargs.get('bottleneck', False):
            _, src = einops.unpack(src, ps, 'b * d')

        # cross-attention decoder
        tgt = self.to_decoder(xi + coo)
        for write in self.decoder:
            tgt = write(tgt, src)

        # prediction head
        tgt = self.to_output(tgt)
        return tgt

In [ ]:
import torch
import einops

from einops.layers.torch import EinMix
from utils.config import *
from utils.components import *
from utils.loss_fn import *

def init_sincos_positions(dim: int, world: WorldConfig):
    # integer indices
    coordinates = torch.stack(torch.unravel_index(indices = torch.arange(world.num_tokens), shape = world.token_shape), dim = -1)
    # log wavelengths
    log_wavelengths = torch.as_tensor(world.token_shape).log()
    # only encode shape dimensions with actual size
    valid = log_wavelengths > 0
    log_wavelengths = log_wavelengths[valid]
    coordinates = coordinates[:, valid]
    # space the frequencies according to the required number of bands
    negative_spacing = torch.linspace(0, -1, dim // (coordinates.size(-1) * 2))
    # calculate the sin/cos embeddings:
    frequencies = torch.exp(negative_spacing * log_wavelengths[..., None])
    angles = torch.einsum("n i, i d -> n i d", coordinates, frequencies) # overflows fp16, be careful
    positions = einops.rearrange([angles.sin(), angles.cos()], 'two n i d -> n (two i d)')
    # avoid uneven dimensions by zero-padding
    positions = torch.nn.functional.pad(positions, (0, dim - positions.size(-1))) 
    return positions

class EinAR(torch.nn.Module):
    def __init__(self, network: NetworkConfig, world: WorldConfig):
        super().__init__()
        # config attributes
        self.network = network
        self.world = world

        # learnable parameters
        self.latent_tokens = torch.nn.Parameter(torch.nn.init.trunc_normal_(torch.zeros(network.num_latents, network.dim), std = network.dim ** -0.5))
        self.src_positions = torch.nn.Parameter(init_sincos_positions(network.dim, world= world))

        # I/O
        self.to_tokens = torch.nn.Sequential(
            EinMix(f'b {world.field_pattern} -> b ({world.token_pattern}) c',
                weight_shape = f'v {world.patch_pattern} c', 
                c = default(network.dim_in, network.dim), 
                **world.token_sizes, **world.patch_sizes),
            EinMix(f'b ({world.token_pattern}) c -> b ({world.token_pattern}) d',
                weight_shape = f'v d c',
                d = network.dim, c = default(network.dim_in, network.dim), 
                **world.token_sizes),
            torch.nn.RMSNorm(network.dim)
        )

        self.to_output = torch.nn.Sequential(
            EinMix(f'b ({world.token_pattern}) d -> (k b) {world.field_pattern}',
                   weight_shape = f'k v {world.patch_pattern} d',
                   d = default(network.dim_out, network.dim), k = network.num_tails, 
                   **world.patch_sizes, **world.token_sizes),
            GaussianSmoothing3D(world.field_shape[0], kernel_size= 5, sigma= 1.),
            Rearrange('(k b) ... -> k b ...', k = network.num_tails)
        )
        
        # Encoder
        self.predictor = torch.nn.ModuleList([
                TransformerBlock(dim= network.dim, drop_path= network.drop_path) 
                for _ in range(default(network.num_layers, 1))
                ])
        
        # weight initialization
        self.apply(self.base_init)
        
    def base_init(self, m: torch.nn.Module):
        if isinstance(m, torch.nn.Linear) or isinstance(m, EinMix):
            torch.nn.init.trunc_normal_(m.weight, std = m.weight.size(-1) ** -0.5)
            if exists(m.bias):
                torch.nn.init.zeros_(m.bias)            
   
    def forward(self, fields: torch.FloatTensor, num_steps: int = 1):
        # tokenize input
        src = self.to_tokens(fields) + self.src_positions
        
        # roll-out
        predictions = []
        for _ in range(num_steps):
            # add cls tokens
            cls = einops.repeat(self.latent_tokens, 'z d -> b z d', b= src.size(0))
            latents, shape = einops.pack([src, cls], 'b * d')
            
            # transformer stack
            for predict in self.predictor:
                latents = predict(latents)
            
            # forward src tokens
            src, cls = einops.unpack(latents, shape, 'b * d')
            
            # decode
            pred = self.to_output(src)
            predictions.append(pred)
        
        return einops.pack(predictions, 'k b v * h w')[0]

In [51]:
network = NetworkConfig(dim = 384, num_latents= 16, num_layers= 8, drop_path= 0.1, num_tails= 2)
world = WorldConfig({'v': 3, 't': 6, 'h': 64, 'w': 120}, patch_sizes= {'vv': 3, 'tt': 1, 'hh': 8, 'ww': 8}, batch_size=4)

In [52]:
ar = EinAR(world=world, network=network)

In [53]:
fields = torch.randn((world.batch_size, *world.field_shape))

In [57]:
test = ar(fields, 4)

In [58]:
test.shape

torch.Size([2, 4, 3, 24, 64, 120])

In [59]:
world.num_tokens

720

In [ ]:
# FORWARD METHODS
    def forward_step(self, batch_idx, batch, step: str = 'train'):
        loss, mu, sigma = self.ar_step(batch_idx, batch)
        if step == 'frcst':
            return mu, sigma
        else:
            return loss
    
    def ar_step(self, batch_idx, batch):        
        # steps
        steps = 1 if self.mode == 'train' and self.step_counter < self.objective.kwargs.get('pre_steps', 1) else self.world.tau

        # work around field size being per step
        mask = einops.repeat(self.land_sea_mask, f'v t h w -> b v (s t) h w', b = batch.size(0), s = steps)
        w_v = einops.repeat(self.per_variable_weights, f'v t h w -> b v (s t) h w', b = batch.size(0), s = steps)

        # split batch
        T = self.world.field_sizes['t']
        src, tgt = batch[:, :, :T], batch[:, :, T: (steps + 1) * T]

        # forward
        prediction = self.model(src, steps)
        
        # loss
        mu, sigma = prediction
        sigma = torch.nn.functional.softplus(sigma)
        loss = f_gaussian_crps(tgt, mu, sigma).mul(w_v)[mask].mean()

        #track metrics
        metrics = {'loss' : loss.item(),
                   'acc': self.compute_acc(mu[mask], tgt[mask]),
                   'rmse': self.compute_rmse(mu[mask], tgt[mask]),
                   'ssr': (sigma[mask].pow(2).mean().sqrt() / (mu[mask] - tgt[mask]).pow(2).mean().sqrt()).item(),
                   }
        self.log_metrics(metrics)

        # update step counter if training
        self.step_counter = self.step_counter + 1 if self.mode == 'train' else self.step_counter
        return loss, mu, sigma
    
    def masked_step(self, batch_idx, batch):        
        # sample masks
        visible, masked = self.sample_masks(batch.size(0))
        
        # foward model
        prediction = self.model(batch, visible)
        
        # sea-only mask
        mask = einops.repeat(masked, f"b ({self.world.token_pattern}) -> b {self.world.field_pattern}", 
                             **self.world.token_sizes, **self.world.patch_sizes)
        mask = torch.logical_and(mask, self.land_sea_mask)

        # loss
        mu, sigma = prediction
        sigma = torch.nn.functional.softplus(sigma)
        loss = f_gaussian_crps(batch, mu, sigma).mul(self.per_variable_weights)[mask].mean()

        #track metrics
        metrics = {'loss' : loss.item(),
                   'acc': self.compute_acc(mu[mask], batch[mask]),
                   'rmse': self.compute_rmse(mu[mask], batch[mask]),
                   'ssr': (sigma[mask].pow(2).mean().sqrt() / (mu[mask] - batch[mask]).pow(2).mean().sqrt()).item(),
                   }
        self.log_metrics(metrics)

        # update step counter if training
        self.step_counter = self.step_counter + 1 if self.mode == 'train' else self.step_counter
        return loss, mu, sigma
    
    def frcst_step(self, batch_idx, batch):
        visible = self.frcst_prefix.expand(batch.size(0), -1)
        prediction = self.model(batch, visible)
        
        # sea-only mask
        mask = einops.repeat(visible.logical_not(), f"b ({self.world.token_pattern}) -> b {self.world.field_pattern}", 
                             **self.world.token_sizes, **self.world.patch_sizes)
        mask = torch.logical_and(mask, self.land_sea_mask)

        # gaussian loss
        mu, sigma = prediction
        mu = mu * self.land_sea_mask
        sigma = torch.nn.functional.softplus(sigma)
        loss = f_gaussian_crps(batch, mu, sigma)[mask].mean()

        metrics = {'frcst_loss' : loss.item(),
                   'frcst_acc': self.compute_acc(mu[mask], batch[mask]),
                   'frcst_rmse': self.compute_rmse(mu[mask], batch[mask]),
                   'frcst_ssr': (sigma[mask].pow(2).mean().sqrt() / (mu[mask] - batch[mask]).pow(2).mean().sqrt()).item(),
                   }
        self.log_metrics(metrics)
        return loss, mu, sigma

    #EVAL
    def evaluate_epoch(self):
        super().evaluate_epoch()
        #self.evaluate_frcst()

    def evaluate_frcst(self):
        self.switch_mode(train=False)
        if not exists(self.val_dl):
            return
        samples = []
        for batch_idx, batch in enumerate(self.val_dl):
            batch = batch.to(self.device)
            with torch.no_grad():
                with torch.amp.autocast(device_type = self.device.type, enabled=self.cfg.mixed_precision):
                    mu, sigma = self.forward_step(batch_idx, batch, 'frcst')
                    samples.append(self.get_xarray_dataset(batch_idx, obs = batch.cpu(), pred = mu[..., None].cpu()))

        ds = xr.concat(samples, dim = "time")
        ds = ds.sel(lat = slice(-20., 20.), lon = slice(90, 270))
        self.get_nino_metrics(ds)
        self.get_field_metrics(ds)

        if self.is_root:
            self.make_eval_plots(ds)

        if self.is_root and self.current_epoch == self.total_epochs and self.cfg.save_eval:
            self.write_to_disk(ds)

    def make_eval_plots(self, ds: xr.Dataset):
        # SAMPLES
        plt.figure(figsize=(12,12))
        plt.subplot(321)
        ds[f"temp_ocn_0a_pred"].isel(time = 0, lag = 20).mean('ens').plot(vmin=-2, vmax = 2, cmap= 'bwr')
        plt.subplot(322)
        ds[f"temp_ocn_0a_tgt"].isel(time = 0, lag = 20).plot(vmin=-2, vmax = 2, cmap= 'bwr')
        # plt.subplot(323)
        # ds[f"temp_ocn_0a_pred"].isel(time = 0, lag = 20, ens = 0).plot(vmin=-2, vmax = 2, cmap= 'bwr')
        # plt.subplot(324)
        # ds[f"temp_ocn_0a_pred"].isel(time = 0, lag = 20, ens = 1).plot(vmin=-2, vmax = 2, cmap= 'bwr')
        # plt.subplot(325)
        # ds[f"temp_ocn_0a_pred"].isel(time = 0, lag = 20, ens = 2).plot(vmin=-2, vmax = 2, cmap= 'bwr')
        # plt.subplot(326)
        # ds[f"temp_ocn_0a_pred"].isel(time = 0, lag = 20, ens = 3).plot(vmin=-2, vmax = 2, cmap= 'bwr')
        plt.savefig(self.model_dir / "test_sample.png")
        plt.close()

        #RANK HIST
        # plt.figure(figsize=(12,4))
        # E = len(ds.ens)
        # ens = ds[f"temp_ocn_0a_pred"].sel(lag = [1, 7, 13, 19]).values.reshape(-1, E)
        # obs = ds[f"temp_ocn_0a_tgt"].sel(lag = [1, 7, 13, 19]).values.reshape(-1, 1)
        # rank_counts = np.bincount(np.sum(ens < obs, axis= -1), minlength= E + 1) / ens.shape[0]
        # plt.bar(np.arange(E + 1), rank_counts, alpha = 0.5)
        # plt.hlines(1 / (E + 1), 0, E, color="red", linestyle="dashed", linewidth=1)
        # plt.ylabel('Frequency')
        # plt.xlabel("Rank")
        # plt.savefig(self.model_dir / "rank_hist.png")
        # plt.close()

        # ACC vs LAG
        plt.figure(figsize=(12,4))
        nino34_tgt, nino34_pred = self.get_nino34(ds["temp_ocn_0a_tgt"]), self.get_nino34(ds["temp_ocn_0a_pred"].mean('ens'))
        nino4_tgt, nino4_pred = self.get_nino4(ds["temp_ocn_0a_tgt"]), self.get_nino4(ds["temp_ocn_0a_pred"].mean('ens'))
        nino34_pcc = self.xr_pcc(nino34_pred, nino34_tgt, ("time",))
        nino4_pcc = self.xr_pcc(nino4_pred, nino4_tgt, ("time",))
        pcc = self.xr_pcc(ds["temp_ocn_0a_pred"].mean('ens'), ds["temp_ocn_0a_tgt"], ('lat', 'lon')).mean(('time'))
        plt.plot(ds.lag, nino34_pcc, label = 'nino3.4')
        plt.plot(ds.lag, nino4_pcc, label = 'nino4')
        plt.plot(ds.lag, pcc, label = 'SSTa')
        plt.ylim(0, 1)
        plt.hlines(0.5, ds.lag[0], ds.lag[-1], colors='r', linestyles='dashed')
        plt.legend()
        plt.xlabel("Lag")
        plt.ylabel("Correlation")
        plt.tight_layout()
        plt.savefig(self.model_dir / "skill.png")
        plt.close()

    def write_to_disk(self, data: xr.Dataset):
        path = self.model_dir / f"{self.data_cfg.eval_data}_eval.zarr"
        data.to_zarr(path, mode = "w")

    def get_xarray_dataset(self, batch_idx, pred, obs):
        #meta data
        meta_data = self.val_dataset.dataset
        time, lat, lon = meta_data.time, meta_data.lat, meta_data.lon
        ens = np.arange(pred.shape[-1])
        tau = self.world.tau
        T, tt = self.world.token_sizes["t"], self.world.patch_sizes["tt"]
        lag = np.arange(1, 1 + ((T - tau) * tt))
        history = tau * tt

        # variables
        arrays = []
        for v, var in enumerate(self.data_cfg.variables):
            if var not in self.data_cfg.eval_variables:
                continue
            
            std = self.val_dataset._stds.sel(variable = var).values
            p = pred[:, v, history:].float().detach().cpu().numpy() * std
            o = obs[:, v, history:].float().detach().cpu().numpy() * std

            #create xarray
            data_array = xr.Dataset(
                data_vars = {
                    f"{var}_pred": (["time", "lag", "lat", "lon", "ens"], p),
                    f"{var}_tgt": (["time", "lag", "lat", "lon"], o),
                },
                coords = {
                    "time": time[batch_idx * self.world.batch_size: (batch_idx + 1) * self.world.batch_size],
                    "lag": lag,
                    "lat": lat,
                    "lon": lon,
                    "ens": ens
                },
            )
            arrays.append(data_array)
        ds = xr.merge(arrays)
        return ds

    def get_xr_lsm(self, data: xr.Dataset):
        if "sftlf" in data:
            lsm = data["sftlf"]
        else:
            lsm = data[self.data_cfg.variables[0]].isel(time=0).isnull()
            lsm = lsm.drop_vars(["time", "month"], errors="ignore")
        return lsm

    def get_field_metrics(self, eval_data: xr.Dataset):
        for var in self.data_cfg.variables:
            if var not in self.data_cfg.eval_variables:
                continue
            tgt, pred = eval_data[f"{var}_tgt"], eval_data[f'{var}_pred']
            pcc = self.xr_pcc(pred.mean('ens'), tgt, ('lat', 'lon')).mean(('time', 'lag'))
            rmse = self.xr_rmse(pred.mean('ens'), tgt, ('lat', 'lon')).mean(('time', 'lag'))
            ssr = self.xr_spread_skill_ens(pred, tgt, ('lat', 'lon')).mean(('time', 'lag'))
            
            self.current_metrics.log_metric(f"{var}_pcc", pcc.item())
            self.current_metrics.log_metric(f"{var}_ssr", ssr.item())
            self.current_metrics.log_metric(f"{var}_rmse", rmse.item())

    def get_nino_metrics(self, eval_data: xr.Dataset):
        nino34_tgt, nino34_pred = self.get_nino34(eval_data["temp_ocn_0a_tgt"]), self.get_nino34(eval_data["temp_ocn_0a_pred"]).mean("ens")
        nino4_tgt, nino4_pred = self.get_nino4(eval_data["temp_ocn_0a_tgt"]), self.get_nino4(eval_data["temp_ocn_0a_pred"]).mean("ens")
        
        nino34_pcc = self.xr_pcc(nino34_pred, nino34_tgt, ("time",))
        nino4_pcc = self.xr_pcc(nino4_pred, nino4_tgt, ("time",))

        nino34_rmse = self.xr_rmse(nino34_pred, nino34_tgt, ("time",))
        nino4_rmse = self.xr_rmse(nino4_pred, nino4_tgt, ("time",))

        nino4_thresh_month =  1 + np.argwhere(nino4_pcc.values > 0.5).max(initial=0)
        nino34_thresh_month = 1 + np.argwhere(nino34_pcc.values > 0.5).max(initial=0)

        self.current_metrics.log_metric('nino4_pcc_month', float(nino4_thresh_month))
        self.current_metrics.log_metric('nino34_pcc_month', float(nino34_thresh_month))

        for lag in [3, 9, 15, 18, 21]:
            self.current_metrics.log_metric(f"nino34_pcc_{lag}", nino34_pcc.sel(lag = lag).item())
            self.current_metrics.log_metric(f"nino34_rmse_{lag}", nino34_rmse.sel(lag = lag).item())
            self.current_metrics.log_metric(f"nino4_pcc_{lag}", nino4_pcc.sel(lag = lag).item())
            self.current_metrics.log_metric(f"nino4_rmse_{lag}", nino4_rmse.sel(lag = lag).item())
        
    def log_metrics(self, metrics: dict, task: str = None):
        for key, val in metrics.items():
            name = f"{task}_{key}" if exists(task) and task != 'prior' else key
            self.current_metrics.log_metric(name, val)
    
    def compute_metrics_torch(self, ens: torch.Tensor, obs: torch.Tensor, mask: torch.BoolTensor):
        ens = ens[mask]
        obs = obs[mask]
        metrics = {
            "crps": self.compute_crps(pred=ens, obs=obs, fair =  self.use_fair_crps).item(),
            "ssr": self.compute_spread_skill(pred=ens, obs=obs).item(),
            "ign": self.compute_ign(pred=ens, obs=obs).item(),
            "spread": self.compute_spread(pred=ens).item(),
            "acc": self.compute_acc(pred=ens.mean(-1), obs=obs).item(),
            "rmse": self.compute_rmse(pred=ens.mean(-1), obs=obs).item(),
        }
        return metrics

    @staticmethod
    def get_nino4(da: xr.DataArray):
        return da.sel(lon=slice(160, 210), lat=slice(-5, 5)).mean(dim=['lon', 'lat'])
    
    @staticmethod
    def get_nino34(da: xr.DataArray):
        return da.sel(lon=slice(190, 240), lat=slice(-5, 5)).mean(dim=['lon', 'lat'])

    @staticmethod
    def xr_pcc(pred: xr.DataArray, obs: xr.DataArray, dim: tuple[str]):
        num = (pred * obs).sum(dim)
        denom = np.sqrt((pred**2).sum(dim)) * np.sqrt((obs**2).sum(dim))
        return num / denom

    @staticmethod
    def xr_rmse(pred: xr.DataArray, obs: xr.DataArray, dim: tuple[str]):
        return np.sqrt(((pred - obs) ** 2).mean(dim))

    @staticmethod
    def xr_spread_skill_ens(pred: xr.DataArray, obs: xr.DataArray, dim: tuple[str]):
        K = pred.sizes["ens"]
        correction = math.sqrt((K + 1) / K)
        mean = pred.mean("ens")
        spread = np.sqrt(pred.var("ens").mean(dim))
        skill = np.sqrt(((obs - mean) ** 2).mean(dim))
        return correction * (spread / skill)
    
    @staticmethod
    def compute_acc(pred, obs, eps = 1e-5)-> float:
        return (pred * obs).nansum().div(pred.pow(2).nansum().sqrt() * obs.pow(2).nansum().sqrt() + eps).item()
    
    @staticmethod
    def compute_rmse(pred, obs)-> float:
        return (pred - obs).pow(2).nanmean().sqrt().item()
    
    @staticmethod
    def compute_crps_ens(pred, obs, fair: bool = True)-> float:
        crps = f_kernel_crps(observation=obs, ensemble=pred, fair = fair)
        return crps.nanmean().item()
    
    @staticmethod
    def compute_ign_ens(pred, obs, eps = 1e-5)-> float:
        ign = f_gaussian_ignorance(observation=obs, mu=pred.mean(-1), sigma=pred.std(-1) + eps)
        return ign.nanmean().item()
    
    @staticmethod
    def compute_spread_ens(pred)-> float:
        return pred.var(-1).mean().sqrt().item()

    @staticmethod
    def compute_spread_skill_ens(pred, obs, eps = 1e-5) -> float:
        K = pred.shape[-1]
        correction = math.sqrt((K + 1) / K)
        mean = pred.mean(-1)
        spread = pred.var(-1).mean().sqrt()
        skill = (obs - mean).pow(2).mean().sqrt() + eps
        return (spread / skill).mul(correction).item()
